# Indic Dubbing — XTTS Synthesis Run

Runs a whole synthesis bundle and prints QC metrics.

Decoder settings come from `colab/xtts_worker.py`, which encodes the findings in
`colab/xtts.md`: greedy decoding for reproducible duration, and short reference
conditioning. Change them there, not here, so the settings stay recorded in the
result artifact.

In [ ]:
!git clone https://github.com/ayushk1233/indic-dub-pipeline.git 2>/dev/null || echo "already cloned"
%cd /content/indic-dub-pipeline
!git pull
!pip install -q -r colab/requirements.txt

In [ ]:
from colab.preflight import PreflightValidator

print(PreflightValidator().run())

Upload the `tts_bundle.zip` produced locally by `BundleExporter`.

In [ ]:
import zipfile
from pathlib import Path

from google.colab import files

uploaded = files.upload()

BUNDLE = Path("/content/tts_bundle")
BUNDLE.mkdir(exist_ok=True)

with zipfile.ZipFile(next(iter(uploaded)), "r") as z:
    z.extractall(BUNDLE)

for p in sorted(BUNDLE.rglob("*")):
    print(p.relative_to(BUNDLE))

## Synthesize every segment

Writes `output/seg_NNNNN.wav` per segment plus `output/synthesis_result.json`,
re-saved after each segment so a timeout still leaves usable work.

In [ ]:
from pathlib import Path

from colab.xtts_worker import XTTSWorker

worker = XTTSWorker(Path("/content/tts_bundle"))
result = worker.run()

## QC report

In [ ]:
from pathlib import Path

from src.eval.harness import build_report, render_report

report = build_report(
    job_dir=Path("/content/tts_bundle"),
    bundle_dir=Path("/content/tts_bundle"),
)

print(render_report(report))

Reading the synthesis table:

- `ratio` above 1.00 means the audio does not fit its slot.
- `tempo` is the time-stretch that would be needed; past about 1.25 it sounds rushed.
- `cps` is the delivered speaking rate. Far below ~13 for Hindi means the model padded.
- `sim` is cosine similarity to the reference speaker. Below 0.75 the clone drifted.
- `tok` near the decoder ceiling means generation never terminated on its own.

## Listen

In [ ]:
from IPython.display import Audio, display

for segment in result.segments[:5]:
    if segment.status != "done":
        continue
    print(f"segment {segment.segment_id}: {segment.duration:.2f}s")
    display(Audio(f"/content/tts_bundle/{segment.audio_path}"))

## Determinism check

Re-synthesize one segment. With greedy decoding the durations must match exactly.

In [ ]:
first = worker.request.segments[0]

a = worker.synthesize_segment(first)
b = worker.synthesize_segment(first)

print(f"run 1: {a.duration:.4f}s, {a.gpt_tokens} tokens")
print(f"run 2: {b.duration:.4f}s, {b.gpt_tokens} tokens")
print("DETERMINISTIC" if a.num_samples == b.num_samples else "STILL SAMPLING — check INFERENCE_PARAMS")

## Download results

In [ ]:
import shutil

from google.colab import files

shutil.make_archive("/content/tts_bundle_out", "zip", "/content/tts_bundle")
files.download("/content/tts_bundle_out.zip")